In [0]:
%python
%pip install geopandas

In [0]:
dbutils.library.restartPython()

In [0]:
import geopandas as gpd
import pandas as pd
from pyspark.sql.functions import lit

In [0]:
# Read TSV
df_tax = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load('/Volumes/cre_catalog/landing/raw_files/nyc_data.tsv')

In [0]:
# Write to bronze
df_tax.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.bronze.nyc_tax_raw")

In [0]:
gdf = gpd.read_file(
    '/Volumes/cre_catalog/landing/raw_files/MAPPLUTO.geojson'
)

In [0]:
%python
# Convert geometry to WKT string for Spark compatibility
gdf['geometry_wkt'] = gdf['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# Drop original geometry column (Spark can't handle geometry type)
gdf_flat = gdf.drop(columns=['geometry'])

# Convert to Spark DataFrame
df_pluto = spark.createDataFrame(gdf_flat)

# Write to bronze
df_pluto.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.bronze.mappluto_raw")

In [0]:
%sql
SELECT 
    COUNT(*) as total_pluto,
    COUNT(t.PARID) as matched_records,
    COUNT(*) - COUNT(t.PARID) as unmatched
FROM cre_catalog.bronze.mappluto_raw p
LEFT JOIN cre_catalog.bronze.nyc_tax_raw t
    ON CAST(p.BBL AS STRING) = t.PARID;